# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Tumour-Board Briefing

### Scenario

You are the visualization engineer in the **3-D Imaging Lab at City General Hospital**. Tomorrow morning the
neuro-oncology **tumour board** (a neurosurgeon, a radiologist and an oncologist) will review a patient's head CT
(`unit3_ppt3_head_ct_phantom.csv`). The lab head forwards you four requests:

1. **IT department:** *"The new scanner protocol is 512 × 512 × 300 at 0.5 × 0.5 × 1.0 mm. How much storage per scan,
   and how many samples does one volume-rendered frame cost?"*
2. **Radiologist:** *"Show me the lesion in all three planes, at its centre, with the anatomy undistorted."*
3. **Neurosurgeon:** *"I need the tumour volume in mL — and I want to know how much that number depends on the
   threshold you picked."*
4. **Oncologist:** *"Your last 3-D render showed a beautiful skull and nothing else. I need to see the tumour and its
   relation to the ventricles."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer in your
own words. The ground-truth `labels` array may be used **only to evaluate** your results, never to find the tumour.

## Part 0 — Setup (given)

Run this cell as-is. It loads the volume `vol[z, y, x]`, the evaluation-only `labels`, the voxel spacing, and two
helpers: `make_tf(points)` builds a transfer function from `(hu, (r, g, b), alpha)` control points, and
`render(volume, tf, axis)` ray-casts the volume with front-to-back compositing (resampled at 2× density).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ct = pd.read_csv("unit3_ppt3_head_ct_phantom.csv")
NZ, NY, NX = ct.z.max() + 1, ct.y.max() + 1, ct.x.max() + 1
vol = ct.intensity_hu.values.reshape(NZ, NY, NX)        # V[z, y, x]  (z = feet->head, y = front->back, x = left->right)
labels = ct.label.values.reshape(NZ, NY, NX)            # ground truth: 0 air, 1 scalp, 2 bone, 3 brain, 4 CSF, 5 tumour
SPACING = (4.0, 3.0, 3.0)                               # voxel size in mm along (z, y, x)
VOXEL_MM3 = np.prod(SPACING)
ASPECT = SPACING[0] / SPACING[2]                        # use when z is on a screen axis

def make_tf(points):
    # points: list of (hu, (r, g, b), alpha) control points -> transfer function T(s) -> (C, alpha)
    hu = np.array([p[0] for p in points], float)
    rgb = np.array([p[1] for p in points], float)
    alpha = np.array([p[2] for p in points], float)
    def tf(s):
        c = np.stack([np.interp(s, hu, rgb[:, k]) for k in range(3)], axis=-1)
        return c, np.interp(s, hu, alpha)
    return tf

def render(volume, tf, axis=1, upsample=2):
    # Orthographic ray casting with front-to-back compositing; rays travel along `axis`.
    # upsample=2 resamples the volume trilinearly at twice the density (smoother, fewer aliasing artifacts).
    v = ndimage.zoom(volume.astype(float), upsample, order=1) if upsample > 1 else volume
    v = np.moveaxis(v, axis, 0)
    C = np.zeros(v.shape[1:] + (3,)); A = np.zeros(v.shape[1:])
    for sample in v:                                    # one sample per voxel step along every ray at once
        c, a = tf(sample)
        C += ((1 - A) * a)[..., None] * c
        A += (1 - A) * a
    return C, A

print(f"vol {vol.shape} (z, y, x), voxel {SPACING} mm, {VOXEL_MM3:.0f} mm^3 per voxel")

vol (48, 64, 64) (z, y, x), voxel (4.0, 3.0, 3.0) mm, 36 mm^3 per voxel


## Task 1 — The IT Department's Numbers (Request #1)

**Real-world usage:** PACS storage and GPU sizing for 3-D workstations are budgeted from exactly these figures.

**Why this technique:** Voxel count = Nx·Ny·Nz (Slides 7, 12); each ray samples the volume at successive positions
(Slide 22), so one frame costs ≈ rays × samples per ray; early ray termination skips samples once α_acc → 1
(Slide 27).

**TODO:**
1. Number of voxels in a 512 × 512 × 300 scan, and its size in MB at 16 bits per voxel.
2. Physical size of the scanned region in mm, and the volume of one voxel in mm³ (0.5 × 0.5 × 1.0 mm).
3. Samples needed for ONE front-view rendered frame (one ray per x–z pixel, one sample per voxel step in y), and
   for a 360° rotation video of 120 frames.
4. If early ray termination stops rays after 35% of their samples on average, how many samples does the 120-frame
   video need instead?

In [ ]:
# TODO 1
n_voxels = 512 * 512 * 300
scan_mb = n_voxels * 2 / 1024**2            # 16 bits = 2 bytes per voxel

# TODO 2
extent_mm = (512 * 0.5, 512 * 0.5, 300 * 1.0)   # (x, y, z) in mm
voxel_mm3 = 0.5 * 0.5 * 1.0

# TODO 3
samples_per_frame = 512 * 300 * 512         # rays (x-z pixels) x samples per ray (y steps)
samples_video = samples_per_frame * 120

# TODO 4
samples_video_ert = samples_video * 0.35

print(f"{n_voxels:,} voxels = {scan_mb:.1f} MB | extent {extent_mm} mm | voxel {voxel_mm3} mm^3")
print(f"samples/frame {samples_per_frame:,} | 120 frames {samples_video:,} | with early termination {samples_video_ert:,.0f}")

**Reflection:** The new protocol has *anisotropic* voxels (0.5 × 0.5 × 1.0 mm). What two things would go wrong in the
radiologist's coronal and sagittal views if the software ignored that (Slides 8–10)?

_Your answer:_ First, the shapes would be distorted. Each slice voxel is 1.0 mm tall in z but only 0.5 mm wide in x and y, so if every voxel is drawn as a square the coronal and sagittal views come out squashed to half their true height. A round skull looks like a flat oval and the lesion looks flatter than it is. Second, any measurement taken on those views would be wrong. Distances, sizes and angles read along z would be off by a factor of 2 (a length drawn as 100 pixels is really 100 mm in z, not 50 mm), so the radiologist could misjudge how big the lesion is or how far it is from a structure. The fix is to display z with the correct aspect ratio, which is what the aspect argument does in Task 2.


## Task 2 — Find the Lesion and Show It in Three Planes (Request #2)

**Real-world usage:** Every radiology workstation opens on a 3-plane MPR with linked cross-hairs.

**Why this technique:** **Orthogonal slices / MPR** give complementary views of the same structure (Slides 15–17);
a **window/level** setting is the slice-display transfer function; the correct **voxel aspect ratio** keeps anatomy
undistorted (Slide 9).

**TODO:**
1. Find the lesion *without* `labels`: take voxels with 55 < HU < 90 that lie **inside the brain** — use
   `inside = ndimage.binary_fill_holes(vol > 300) & ~(vol > 300)` and erode it 3 times
   (`ndimage.binary_erosion(..., iterations=3)`) to avoid the partial-volume shell under the skull.
2. Keep the largest connected component (`ndimage.label`) and compute its centroid (z, y, x) as integers.
3. Plot axial, coronal and sagittal slices through that centroid with cross-hairs, a brain window (W = 80, L = 40),
   and `aspect=ASPECT` for the two views that contain z.
4. Evaluation only: print how far (in mm) your centroid is from the centroid of `labels == 5`.

In [ ]:
def window(a, W, L):
    return np.clip((a - (L - W / 2)) / W, 0, 1)

# TODO 1
inside = ndimage.binary_fill_holes(vol > 300) & ~(vol > 300)
inside = ndimage.binary_erosion(inside, iterations=3)
candidates = inside & (vol > 55) & (vol < 90)

# TODO 2
comp, n_comp = ndimage.label(candidates)
sizes = ndimage.sum(candidates, comp, range(1, n_comp + 1))
lesion = comp == (1 + np.argmax(sizes))
cz, cy, cx = [int(round(v)) for v in ndimage.center_of_mass(lesion)]
print(f"{n_comp} candidate region(s); lesion has {lesion.sum()} voxels, centroid (z, y, x) = ({cz}, {cy}, {cx})")

# TODO 3
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
W, L = 80, 40
# axial: plane z = cz, shown as (y, x)
axes[0].imshow(window(vol[cz], W, L), cmap="gray", vmin=0, vmax=1)
axes[0].axhline(cy, color="yellow", lw=0.8); axes[0].axvline(cx, color="yellow", lw=0.8)
axes[0].set_title(f"Axial (z = {cz})"); axes[0].set_xlabel("x (left to right)"); axes[0].set_ylabel("y (front to back)")
# coronal: plane y = cy, shown as (z, x), z points up
axes[1].imshow(window(vol[:, cy, :], W, L), cmap="gray", vmin=0, vmax=1, aspect=ASPECT, origin="lower")
axes[1].axhline(cz, color="yellow", lw=0.8); axes[1].axvline(cx, color="yellow", lw=0.8)
axes[1].set_title(f"Coronal (y = {cy})"); axes[1].set_xlabel("x (left to right)"); axes[1].set_ylabel("z (feet to head)")
# sagittal: plane x = cx, shown as (z, y), z points up
axes[2].imshow(window(vol[:, :, cx], W, L), cmap="gray", vmin=0, vmax=1, aspect=ASPECT, origin="lower")
axes[2].axhline(cz, color="yellow", lw=0.8); axes[2].axvline(cy, color="yellow", lw=0.8)
axes[2].set_title(f"Sagittal (x = {cx})"); axes[2].set_xlabel("y (front to back)"); axes[2].set_ylabel("z (feet to head)")

plt.tight_layout(); plt.show()

# TODO 4 (evaluation only)
true_c = np.array(ndimage.center_of_mass(labels == 5))
err_mm = np.linalg.norm((np.array([cz, cy, cx]) - true_c) * np.array(SPACING))
print(f"Distance from the true tumour centroid: {err_mm:.1f} mm")

**Reflection:** Why did you need the erosion step? Explain in terms of the **partial-volume effect** and the
overlapping intensity ranges the lecture lists as a transfer-function challenge (Slide 28).

_Your answer:_ The brain is surrounded by a shell where each voxel covers both bone (very bright) and brain (about 30 to 45 HU), so the scanner stores an average of the two. This is the partial-volume effect. As we move from bone to brain, those mixed voxels pass through every value in between, including the 55 to 90 HU range that I used for the tumour, so they look exactly like tumour by intensity alone. This is the overlapping intensity range problem from Slide 28: different tissues, or a boundary between tissues, can share the same HU value, so a threshold cannot tell them apart. Eroding the brain mask 3 times removes the shell under the skull, so the only voxels left in the 55 to 90 HU range are real tumour, and the result is one clean region of 431 voxels.


## Task 3 — Tumour Volume and Its Threshold Sensitivity (Request #3)

**Real-world usage:** Tumour volume drives treatment decisions and response assessment, so its uncertainty must be
reported.

**Why this technique:** An **isosurface/segmentation at threshold c** defines the structure — and *the chosen c
strongly influences the resulting geometry* (Slides 19–20, 31–32).

**TODO:**
1. For c in `[45, 50, 55, 60, 65, 70]`, grow the region `vol > c` from your lesion centre (connected component
   containing `(cz, cy, cx)`).
2. For each c, record the voxel count, the volume in mL (`voxels * VOXEL_MM3 / 1000`), and — evaluation only — the
   Dice overlap with `labels == 5`.
3. Plot volume (mL) vs. threshold, and mark the threshold with the best Dice.
4. Print the range of volumes you would report to the surgeon.

In [ ]:
truth = labels == 5
rows = []
# TODO 1 & 2
for c in [45, 50, 55, 60, 65, 70]:
    grown, _ = ndimage.label(vol > c)
    region = grown == grown[cz, cy, cx]               # connected component containing the lesion centre
    n = int(region.sum())
    dice = 2 * (region & truth).sum() / (region.sum() + truth.sum())
    rows.append({"threshold_HU": c, "voxels": n, "volume_mL": round(n * VOXEL_MM3 / 1000, 2), "dice": round(float(dice), 3)})

table = pd.DataFrame(rows)
display(table)

# TODO 3
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(table.threshold_HU, table.volume_mL, "o-", color="tab:blue")
best = table.loc[table.dice.idxmax()]
ax.axvline(best.threshold_HU, color="red", ls="--")
ax.plot(best.threshold_HU, best.volume_mL, "o", color="red", ms=10)
ax.annotate(f"best Dice {best.dice:.2f} at {best.threshold_HU:.0f} HU", (best.threshold_HU, best.volume_mL),
            xytext=(8, 10), textcoords="offset points", color="red")
ax.set_xlabel("Threshold (HU)"); ax.set_ylabel("Tumour volume (mL)")
ax.set_title("The reported tumour volume falls as the threshold rises")
plt.tight_layout(); plt.show()

# TODO 4
lo, hi = table.volume_mL.min(), table.volume_mL.max()
print(f"Report to the surgeon: about {lo:.1f} to {hi:.1f} mL over thresholds 45 to 70 HU "
      f"(best single estimate {best.volume_mL:.1f} mL at {best.threshold_HU:.0f} HU).")

**Reflection:** In real life there is no ground-truth label. How would you choose and justify c for the tumour
board, and how would you *visualize* the uncertainty so the surgeon doesn't treat a single number as exact?

_Your answer:_ Without ground truth I would not trust one number. I would pick c from the intensity histogram, at the valley between the brain peak (about 35 HU) and the tumour peak (about 65 HU), and then check on the slices that the contour follows the edge that the radiologist sees. Here the table shows why this matters: the volume goes from about 19 mL at 45 HU to about 6 mL at 70 HU, so a small change in c changes the answer by a factor of 3. The best threshold only gave 16.7 mL because I could compare against the labels. To show the uncertainty I would give the volume as a range (about 6 to 19 mL), and draw the contours for several thresholds as nested outlines on the MPR slices, or a plot of volume against threshold like the one above. If the outlines are close together the edge is sharp and the number is reliable, and if they are far apart the surgeon can see the number is soft.


## Task 4 — A 3-D Render That Shows the Tumour (Request #4)

**Real-world usage:** Pre-operative 3-D views show a lesion in context with critical structures (here the ventricles).

**Why this technique:** **Direct volume rendering** with a designed **transfer function** — colour to *identify*
materials, opacity to *reveal or suppress* them (Slides 21–27); occlusion is the main enemy (Slide 28).

**TODO:**
1. Render the given `bone_tf` (front view, `axis=1`) — this is the "beautiful skull" the oncologist complained about.
2. Design your own `tumour_tf` with `make_tf([...])` so that: air and soft tissue are fully transparent, bone is
   *almost* transparent (α ≤ 0.01, still gives context), CSF (≈ 5–20 HU) is blue and faint, and the tumour range
   is red. Remember that opacity **accumulates over every sample along a ray**: a thick structure with α ≈ 0.1
   per sample becomes nearly opaque, while a thin one stays faint — use that to your advantage.
3. Render your TF from the front (`axis=1`) **and** from above (`axis=0`), side by side with the bone render.
4. Give the figure a conclusion-style title.

In [ ]:
bone_tf = make_tf([(-1000, (0, 0, 0), 0), (200, (0, 0, 0), 0), (400, (1, 0.95, 0.85), 0.3), (1500, (1, 1, 1), 0.6)])

# TODO 2: your transfer function
tumour_tf = make_tf([
    (-1000, (0, 0, 0), 0.0),        # air: transparent
    (-300,  (0, 0, 0), 0.0),        # scalp fat: transparent
    (-1,    (0, 0, 0), 0.0),
    (2,     (0.2, 0.4, 1.0), 0.0),  # CSF range starts: blue
    (5,     (0.2, 0.4, 1.0), 0.04),
    (20,    (0.2, 0.4, 1.0), 0.04), # CSF 5 to 20 HU: blue and faint
    (24,    (0.2, 0.4, 1.0), 0.0),
    (45,    (0, 0, 0), 0.0),        # brain tissue (about 30 to 45 HU): transparent
    (50,    (1.0, 0.1, 0.1), 0.0),
    (56,    (1.0, 0.1, 0.1), 0.06), # tumour range 55 to 80 HU: red, strong
    (80,    (1.0, 0.1, 0.1), 0.06),
    (90,    (1.0, 0.1, 0.1), 0.0),
    (200,   (0, 0, 0), 0.0),
    (400,   (1.0, 0.95, 0.85), 0.003),  # bone: almost transparent, only for context
    (1500,  (1.0, 1.0, 1.0), 0.006),
])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
# TODO 1 & 3: bone render | your TF front view | your TF top view
def show(ax, tf, axis, title):
    C, A = render(vol, tf, axis=axis)
    img = np.clip(C + (1 - A)[..., None] * 0.0, 0, 1)         # composite over a black background
    ax.imshow(img, aspect=ASPECT if axis == 1 else 1.0, origin="lower" if axis == 1 else "upper")
    ax.set_title(title); ax.axis("off")
show(axes[0], bone_tf, 1, "Bone transfer function, front view")
show(axes[1], tumour_tf, 1, "Tumour transfer function, front view")
show(axes[2], tumour_tf, 0, "Tumour transfer function, from above")

# TODO 4: suptitle
fig.suptitle("The tumour (red) sits beside the ventricles (blue), which the skull-only render hides", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

**Reflection:** Your tumour TF probably also lights up a faint red rim just inside the skull. Which volume-rendering
challenge from Slide 28 causes it? Name one technique from this lecture (hint: Slides 31–32) that would remove it,
and explain why a transfer function alone cannot.

_Your answer:_ The rim is caused by the partial-volume effect together with overlapping intensity ranges (Slide 28). The voxels between the skull and the brain are averages of bone and brain, so they pass through the same HU range as the tumour and the transfer function colours them red. A transfer function only sees the value of one sample, so it cannot tell a real tumour voxel from a boundary voxel with the same value. Segmentation (Slides 31 and 32) would remove it: find the tumour region first, using a threshold, connected component and the brain mask as in Task 2 and 3, and render only that region (or give it its own colour and opacity). This works because segmentation uses position and connectivity as well as intensity, which a transfer function does not have.


## Task 5 — The Decision (No Code)

**Reflection (final):** Write a 4–6 sentence note to the tumour board covering the whole notebook. It should:
- State where the lesion is and its volume **as a range** (Task 3), in plain language.
- Recommend which **one** view to open the meeting with (MPR, segmentation overlay, or the 3-D render) and justify
  it with at least one lecture idea (occlusion, slicing exposes internal structure, transfer-function overlap,
  segmentation enables measurement).
- Name one limitation of the 3-D render the board should keep in mind.

_Your final note here:_ The lesion sits in the right back part of the brain (the voxel at z = 28, y = 40, x = 41, found from intensity alone and within 3 mm of the true centre), and its volume is between about 6 and 19 mL depending on the threshold, with about 16.7 mL as the best single estimate. I would open the meeting with the three plane MPR view, because slicing exposes the internal structure directly and the lesion can be seen at its true size and position without anything in front of it, whereas the 3-D render gives a nice overview but hides depth. I would follow with the segmentation overlay, because segmentation is what makes measurement possible, and I would show the volume as a range, not as a single exact number, since the result changes a lot with the threshold. The limitation of the 3-D render to keep in mind is that it depends on the transfer function I designed: a different opacity setting would show different structures, and the faint red rim under the skull is a rendering artifact (overlapping intensity ranges), not a second lesion.
